# Was DPoser trained on the AMASS of PoseD-Flow?

Before we compare RSGM with the pretrained DPoser (Lu et al., ICCV 2025; github.com/moonbow721/DPoser-X), check that its
training data is the data of `explore/training/rsgm_posedflow_train.ipynb`: `~/data/posedflow_amass` (the Google Drive
file `1TQi_wKxJU3TTcVko-oPlWvp8L12lNR7F`, linked by the READMEs of both PoseD-Flow and DPoser-X), which
`scripts/preprocess/convert_posedflow.py` turns into the split `posedflow`.

The same link is not proof: their preprocessing (`lib/data/prepare_data.py`) draws the frames of every sequence at
random (`np.random.choice` within 10–90% of it), so only this very dump counts, and the checkpoint on Hugging Face came
with DPoser-X (2025), two years after the dump (August 2023). The checkpoint does not record its data either: its
`hyper_parameters` keep the config and the folder of the normalizer, not `--data-root` or `--version`.

A side note: their preprocessing reads `*_stageii.npz`, the SMPL-X fits of AMASS, and keeps `poses[:, 3:66]`, the 21
body joints of SMPL-X (not the SMPL fits).

The repository keeps some files computed from their data. We compare them with ours:

- **A. The normalizers.** z-score (`axis_normalize2.pt`: `mean_poses`, `std_poses`) and min-max (`axis_normalize1.pt`:
  `min_poses`, `max_poses`) of the 63 axis-angle coordinates, in two places: `body_normalizer/` (the one the demo and
  training use; its readme says only "Splitted as version2") and `version1/train/`.
- **B1. The FID statistics** `version1/statistics.npz`: `mu` and `sigma = np.cov` of the raw axis-angle poses of the
  version1 **train** set (`lib/data/build_faiss.py`, `prepare_body`).
- **B2. The reference batch** `version1/reference_batch.pt` (the real poses of their precision/recall): found row by row
  in our train / valid / test?

A match of the statistics to ~1e-6 means they were computed on exactly our dump; finding the reference poses in it
says the same, pose by pose. What it does **not** prove is that the checkpoint was trained on version1 (check C, on
the checkpoint itself, and D, their Tab. 1 reproduced, come later).

Run the cells in order. Reading the train split (18.6M poses, 4.7 GB, memory-mapped) takes a few minutes; its
moments are cached next to the downloads.

In [1]:
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import torch

# ---- parameters ----------------------------------------------------------------------
DATA = Path("~/data/posedflow_amass").expanduser()  # train/, valid/, test/ with pose_body.pt [N, 63]
DEST = Path("~/data/pretrained/dposer").expanduser()  # the downloads (and a cache of our moments)
COMMIT = "c373fce3d364a4a0946e8445fdea5cbfd490e837"  # main of moonbow721/DPoser-X on 2025-10-08
FILES = [  # paths in their repository
    "data/body_data/body_normalizer/readme.txt",
    "data/body_data/body_normalizer/axis_normalize1.pt",
    "data/body_data/body_normalizer/axis_normalize2.pt",
    "data/body_data/version1/train/axis_normalize1.pt",
    "data/body_data/version1/train/axis_normalize2.pt",
    "data/body_data/version1/statistics.npz",
    "data/body_data/version1/reference_batch.pt",
]
SPLITS = ("train", "valid", "test")
MATCH_TOL = 1e-4  # max |difference| per coordinate for a pose to count as found (radians)
# ----------------------------------------------------------------------------------------

## 1. Download


Seven small files (about 2.6 MB) from GitHub at the pinned commit, into `DEST/github/<their path>`. Files already
there are not downloaded again. Nothing of their code is downloaded or run; the `.pt` files are read with
`torch.load(weights_only=True)` (tensors and plain containers only), the `.npz` with `allow_pickle=False`.

In [2]:
RAW = f"https://raw.githubusercontent.com/moonbow721/DPoser-X/{COMMIT}/"
local = {}
for f in FILES:
    path = DEST / "github" / f
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(RAW + f, timeout=60) as r:
            path.write_bytes(r.read())
    local[f] = path
    print(f"{path.stat().st_size:>10,} B  sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:16]}  {f}")

        20 B  sha256 d29e2656e44374ec  data/body_data/body_normalizer/readme.txt
     1,383 B  sha256 1aa7ba3f53957fd5  data/body_data/body_normalizer/axis_normalize1.pt
     1,383 B  sha256 1d42f3594c9ccd7e  data/body_data/body_normalizer/axis_normalize2.pt
     1,447 B  sha256 5b5ce4b495645fa7  data/body_data/version1/train/axis_normalize1.pt
     1,447 B  sha256 b3b14d77b7d6ed3d  data/body_data/version1/train/axis_normalize2.pt
    32,504 B  sha256 ffaec6db044f8994  data/body_data/version1/statistics.npz
 2,520,747 B  sha256 e59254bb7ce3eb36  data/body_data/version1/reference_batch.pt


## 2. What the files contain

Their key names and shapes (the cells below expect `mean_poses`/`std_poses`, `min_poses`/`max_poses`, `mu`/`sigma` and a
`[N, 63]` tensor; if they print something else, stop here).

In [3]:
def describe(obj, indent="    "):
    if isinstance(obj, dict):
        for k, v in obj.items():
            print(f"{indent}{k}: {tuple(v.shape)} {v.dtype}" if hasattr(v, "shape") else f"{indent}{k}: {type(v).__name__}")
    elif hasattr(obj, "shape"):
        print(f"{indent}{tuple(obj.shape)} {obj.dtype}")
    else:
        print(f"{indent}{type(obj).__name__}")


print("readme:", local["data/body_data/body_normalizer/readme.txt"].read_text().strip())
theirs = {}
for f, path in local.items():
    if path.suffix == ".pt":
        theirs[f] = torch.load(path, map_location="cpu", weights_only=True)
    elif path.suffix == ".npz":
        with np.load(path, allow_pickle=False) as z:
            theirs[f] = {k: z[k] for k in z.files}
    else:
        continue
    print(f)
    describe(theirs[f])

readme: Splitted as version2
data/body_data/body_normalizer/axis_normalize1.pt
    min_poses: (63,) torch.float32
    max_poses: (63,) torch.float32
data/body_data/body_normalizer/axis_normalize2.pt
    mean_poses: (63,) torch.float32
    std_poses: (63,) torch.float32
data/body_data/version1/train/axis_normalize1.pt
    min_poses: (63,) torch.float32
    max_poses: (63,) torch.float32
    min_shapes: NoneType
    max_shapes: NoneType
data/body_data/version1/train/axis_normalize2.pt
    mean_poses: (63,) torch.float32
    std_poses: (63,) torch.float32
    mean_shapes: NoneType
    std_shapes: NoneType
data/body_data/version1/statistics.npz
    mu: (63,) float32
    sigma: (63, 63) float64
data/body_data/version1/reference_batch.pt
    (10000, 63) torch.float32


## 3. The moments of our data

Per split, in float64 over chunks of the memory-mapped tensor: the number of poses, sums, the matrix of products,
min and max. From them: the mean, the covariance (`np.cov`, divided by N − 1), the standard deviation both ways
(`torch.std`: N − 1; `np.std`: N). Also train + valid, in case a normalizer was fitted on both.

In [4]:
def load_split(split):
    return torch.load(DATA / split / "pose_body.pt", mmap=True, weights_only=True)


def raw_moments(x, chunk=1_000_000):
    d = x.shape[1]
    m = {"n": 0, "sum": np.zeros(d), "outer": np.zeros((d, d)), "min": np.full(d, np.inf), "max": np.full(d, -np.inf)}
    for i in range(0, len(x), chunk):
        c = x[i:i + chunk].numpy().astype(np.float64)
        m["n"] += len(c)
        m["sum"] += c.sum(0)
        m["outer"] += c.T @ c
        m["min"] = np.minimum(m["min"], c.min(0))
        m["max"] = np.maximum(m["max"], c.max(0))
    return m


def finish(m):
    n = m["n"]
    mean = m["sum"] / n
    cov = (m["outer"] - n * np.outer(mean, mean)) / (n - 1)
    var = np.diag(cov)
    return {"n": n, "mean": mean, "cov": cov, "std_n-1": np.sqrt(var), "std_n": np.sqrt(var * (n - 1) / n),
            "min": m["min"], "max": m["max"]}


cache = DEST / "posedflow_amass_moments.npz"
if cache.is_file():
    with np.load(cache, allow_pickle=False) as z:
        raw = {s: {k.split("/", 1)[1]: z[k] for k in z.files if k.startswith(s + "/")} for s in SPLITS}
    for s in SPLITS:
        raw[s]["n"] = int(raw[s]["n"])
else:
    raw = {s: raw_moments(load_split(s)) for s in SPLITS}
    np.savez(cache, **{f"{s}/{k}": np.asarray(v) for s in SPLITS for k, v in raw[s].items()})

both = {"n": raw["train"]["n"] + raw["valid"]["n"], "sum": raw["train"]["sum"] + raw["valid"]["sum"],
        "outer": raw["train"]["outer"] + raw["valid"]["outer"],
        "min": np.minimum(raw["train"]["min"], raw["valid"]["min"]),
        "max": np.maximum(raw["train"]["max"], raw["valid"]["max"])}
ours = {s: finish(raw[s]) for s in SPLITS}
ours["train+valid"] = finish(both)
for s, o in ours.items():
    print(f"{s:>12}: {o['n']:>11,} poses")

       train:  18,611,047 poses
       valid:     860,080 poses
        test:      54,200 poses
 train+valid:  19,471,127 poses


## 4. Check A: the normalizers

The largest difference over the 63 coordinates, for every normalizer against every set of ours. A match is
~1e-6 or below (float32 rounding); a different random draw of frames from the same sequences gives about 1e-4 to 1e-3
for the train set, more for the small splits.

In [5]:
def flat(t):
    return np.asarray(t, dtype=np.float64).reshape(-1)


def max_diff(a, b):
    return float(np.abs(flat(a) - flat(b)).max())


rows = []
for f in [f for f in FILES if "axis_normalize" in f]:
    t = theirs[f]
    for s, o in ours.items():
        if "mean_poses" in t:
            rows.append((f, s, "mean", max_diff(t["mean_poses"], o["mean"])))
            rows.append((f, s, "std N-1", max_diff(t["std_poses"], o["std_n-1"])))
            rows.append((f, s, "std N", max_diff(t["std_poses"], o["std_n"])))
        if "min_poses" in t:
            rows.append((f, s, "min", max_diff(t["min_poses"], o["min"])))
            rows.append((f, s, "max", max_diff(t["max_poses"], o["max"])))

print(f"{'their file':<52} {'ours':>12} {'stat':>8} {'max |diff|':>11}")
for f, s, stat, d in rows:
    print(f"{f.replace('data/body_data/', ''):<52} {s:>12} {stat:>8} {d:>11.2e}{'   <- match' if d < 1e-5 else ''}")

their file                                                   ours     stat  max |diff|
body_normalizer/axis_normalize1.pt                          train      min    3.78e+00
body_normalizer/axis_normalize1.pt                          train      max    4.21e+00
body_normalizer/axis_normalize1.pt                          valid      min    4.82e+00
body_normalizer/axis_normalize1.pt                          valid      max    3.14e+00
body_normalizer/axis_normalize1.pt                           test      min    5.37e+00
body_normalizer/axis_normalize1.pt                           test      max    3.27e+00
body_normalizer/axis_normalize1.pt                    train+valid      min    3.78e+00
body_normalizer/axis_normalize1.pt                    train+valid      max    4.21e+00
body_normalizer/axis_normalize2.pt                          train     mean    4.60e-02
body_normalizer/axis_normalize2.pt                          train  std N-1    7.67e-02
body_normalizer/axis_normalize2.pt         

## 5. Check B1: the FID statistics of version1



`mu` and `sigma` against the mean and the covariance (N − 1) of each set of ours. They were computed on the version1
train set, so a match is expected with our train set only.

In [6]:
st = theirs["data/body_data/version1/statistics.npz"]
print(f"{'ours':>12} {'max |d mu|':>11} {'max |d sigma|':>14} {'rel. Frobenius d sigma':>23}")
for s, o in ours.items():
    dmu, dsig = max_diff(st["mu"], o["mean"]), max_diff(st["sigma"], o["cov"])
    rel = np.linalg.norm(st["sigma"] - o["cov"]) / np.linalg.norm(o["cov"])
    print(f"{s:>12} {dmu:>11.2e} {dsig:>14.2e} {rel:>23.2e}{'   <- match' if max(dmu, dsig) < 1e-5 else ''}")

        ours  max |d mu|  max |d sigma|  rel. Frobenius d sigma
       train    7.58e-02       6.95e-02                2.34e-01
       valid    1.37e-01       8.84e-02                4.22e-01
        test    3.61e-01       1.30e-01                6.04e-01
 train+valid    7.77e-02       6.92e-02                2.37e-01


## 6. Check B2: the reference batch, pose by pose

Every reference pose is looked up in each split of ours. The poses are put in cells of 1e-3 rad by their first three
coordinates (with the neighbouring cells, so that rounding at a border loses nothing); only the poses of the same cell
are compared in full. A pose counts as found when all 63 coordinates agree within `MATCH_TOL`.

**What to expect.** A reference batch from our very dump is found in full (~100%, with a difference of exactly 0).
Another run of their preprocessing keeps another 30% of the frames of every sequence (`keep_rate = 0.3`, without
replacement), but their loop reads the sequences of the earlier datasets again for every next dataset of a split
(`npz_fnames` grows inside the loop over the datasets), so a sequence is drawn k times, k up to the number of
datasets, and a pose of another run is in ours with the probability 1 − 0.7^k: anything from 30% to ~97%. A share
below 100% therefore says *another run*, not how different it is; section 7 tells apart frames we did not draw from
poses of other motions. The repeated draws are also why about 40% of our training poses have an exact copy elsewhere
in the split, so the index of a match says nothing.
A test on our own data (5000 drawn training poses and the frames right after them): every one found at 1e-4, 1e-5 and
1e-6 alike, so the tolerance hardly matters.

The batch may be stored normalized: it is tried as it is and denormalized with each normalizer of check A (z-score:
x · std + mean; min-max: (x + 1) / 2 · (max − min) + min). Its own mean and std (printed first) tell which is likely:
raw axis-angle has means of a few tenths of a radian, a z-scored batch means ≈ 0 and stds ≈ 1.

In [7]:
ref = np.asarray(theirs["data/body_data/version1/reference_batch.pt"], dtype=np.float64)
ref = ref.reshape(len(ref), -1)
assert ref.shape[1] == 63, f"the reference batch is {ref.shape}, not [N, 63]: look at it before matching"
print(f"reference batch {ref.shape}: mean of the coordinates in [{ref.mean(0).min():.3f}, {ref.mean(0).max():.3f}], "
      f"std in [{ref.std(0).min():.3f}, {ref.std(0).max():.3f}]")

variants = {"as stored": ref}
for f in [f for f in FILES if "axis_normalize" in f]:
    t, name = theirs[f], f.replace("data/body_data/", "").replace("/axis_normalize", " ").replace(".pt", "")
    if "mean_poses" in t:
        variants[f"z-score^-1 {name}"] = ref * flat(t["std_poses"]) + flat(t["mean_poses"])
    if "min_poses" in t:
        variants[f"min-max^-1 {name}"] = (ref + 1) / 2 * (flat(t["max_poses"]) - flat(t["min_poses"])) + flat(t["min_poses"])

CELL, OFF, BASE = 1e3, 1 << 12, 1 << 13  # cells of 1e-3 rad; |axis-angle| < pi < 4.096


def cell_key(xyz):
    k = np.rint(np.asarray(xyz, dtype=np.float64) * CELL).astype(np.int64) + OFF
    return (k[..., 0] * BASE + k[..., 1]) * BASE + k[..., 2]


def find(x, refs, chunk=2_000_000):
    # index of x for every row of every reference variant (-1: not found), and the largest difference of the found
    buckets = {}
    shifts = np.stack(np.meshgrid(*[[-1, 0, 1]] * 3, indexing="ij"), -1).reshape(-1, 3) / CELL
    for v, r in enumerate(refs):
        for i, k in enumerate(cell_key(r[:, None, :3] + shifts).reshape(len(r), -1)):
            for kk in set(k.tolist()):
                buckets.setdefault(kk, []).append((v, i))
    keys = np.fromiter(buckets, dtype=np.int64)
    found = [np.full(len(r), -1, dtype=np.int64) for r in refs]
    exact = [np.zeros(len(r), dtype=bool) for r in refs]
    worst = [0.0] * len(refs)
    for start in range(0, len(x), chunk):
        c = x[start:start + chunk].numpy().astype(np.float64)
        ck = cell_key(c[:, :3])
        for h in np.nonzero(np.isin(ck, keys))[0]:
            for v, i in buckets[int(ck[h])]:
                if found[v][i] < 0:
                    d = np.abs(c[h] - refs[v][i]).max()
                    if d < MATCH_TOL:
                        found[v][i], exact[v][i], worst[v] = start + h, d == 0, max(worst[v], d)
    return found, exact, worst


names, refs = list(variants), list(variants.values())
print(f"{'reference':<34} {'split':>6} {'found':>16} {'of them exact':>14} {'max |diff| found':>17}")
for s in SPLITS:
    found, exact, worst = find(load_split(s), refs)
    for name, fd, ex, w in zip(names, found, exact, worst):
        k = int((fd >= 0).sum())
        print(f"{name:<34} {s:>6} {k:>6,} / {len(fd):<6,} ({k / len(fd):4.0%}) {int(ex.sum()):>8,} {w:>17.1e}")

reference batch (10000, 63): mean of the coordinates in [-0.780, 0.806], std in [0.022, 0.601]
reference                           split            found  of them exact  max |diff| found
as stored                           train  6,912 / 10,000 ( 69%)    6,829           9.9e-05
min-max^-1 body_normalizer 1        train      0 / 10,000 (  0%)        0           0.0e+00
z-score^-1 body_normalizer 2        train      0 / 10,000 (  0%)        0           0.0e+00
min-max^-1 version1/train 1         train      0 / 10,000 (  0%)        0           0.0e+00
z-score^-1 version1/train 2         train      0 / 10,000 (  0%)        0           0.0e+00
as stored                           valid      0 / 10,000 (  0%)        0           0.0e+00
min-max^-1 body_normalizer 1        valid      0 / 10,000 (  0%)        0           0.0e+00
z-score^-1 body_normalizer 2        valid      0 / 10,000 (  0%)        0           0.0e+00
min-max^-1 version1/train 1         valid      0 / 10,000 (  0%)        0    

## 7. The poses not found: other frames of the same motions, or other motions?

The distance (L2 over the 63 coordinates) of 500 reference poses that are not in our train to their nearest training
pose of ours, against the same for 500 of our own training poses (excluding exact copies, so the nearest one is a
neighbouring frame of the same sequence). About the same distances: frames of our sequences that our run did not draw.
Distances of a different pose (tenths of a radian and more): motions that our dump does not have.

In [ ]:
train = load_split("train")
found, _, _ = find(train, [ref])
not_found = found[0] < 0
rng = np.random.default_rng(0)
q_theirs = torch.from_numpy(ref[not_found][rng.choice(not_found.sum(), min(500, not_found.sum()), replace=False)]).float()
q_ours = train[torch.from_numpy(rng.choice(len(train), 500, replace=False))].float()


def nearest(q, skip=1e-6, chunk=500_000):
    best, qq = torch.full((len(q),), float("inf")), (q * q).sum(1, keepdim=True)
    for i in range(0, len(train), chunk):
        c = train[i:i + chunk].float()
        d2 = (qq + (c * c).sum(1)[None] - 2 * q @ c.T).clamp_min(0)
        d2[d2 < skip ** 2] = float("inf")  # exact copies (and the pose itself)
        best = torch.minimum(best, d2.min(1).values)
    return best.sqrt().numpy()


print(f"not found: {not_found.mean():.0%} of the reference batch")
for name, q in [("our train poses -> the rest of our train", q_ours), ("their poses not found -> our train", q_theirs)]:
    print(f"{name:<42} 10/50/90th percentiles {np.percentile(nearest(q), [10, 50, 90]).round(4)}")

## 8. Verdict (run of 2026-11-02)

- **B2:** 69% of the reference poses of version1 are in our train, 6,829 of them bit for bit; none in our valid or
  test. The same AMASS files and the same preprocessing, the same split by datasets.
- **7:** the poses not found lie at the distance of a neighbouring frame (median 0.035 against 0.019 between our own
  frames; 90th percentile 0.10 against 0.08), not of another motion: frames of our sequences that our run did not draw.
- **B1:** but the statistics differ far beyond a new draw of frames: the mean by up to 0.08 rad (wrists, elbows,
  shoulders), sigma by 23% (relative Frobenius) against our train. The reference batch agrees with `statistics.npz`
  (sigma 6%) and not with us (25%).
- **A:** the normalizer of the checkpoint (`body_normalizer`, "version2") matches neither our train (mean 0.046, std
  0.077) nor version1 (mean 0.087 against `statistics.npz`).

**So:** three different dumps of the same AMASS datasets: ours (= PoseD-Flow's), version1, and version2 (the
normalizer of the checkpoint, probably its training data). They share the motions but weigh them differently.
A likely reason (not checked): with the repeated reads above, a dataset is drawn as many times as datasets come after
it, and their order is `list(set(...))` of strings, which changes with Python's hash seed from run to run.

**For the comparison:** the released DPoser has seen our motions, not our frames, and in another mixture. Metrics on
the test split (Transitions + SSM, seen by no model) stay fair; dNN against our train does not (DPoser is at a
disadvantage). A strictly fair row needs DPoser trained on our dump.